# Non-Housing Services Inflation from BEA Line Items

This notebook constructs a weighted non-housing services inflation series from
the line items listed in `supercore_lines.xlsx`.

It uses:

- BEA Table 2.4.4U (`U20404`) for monthly price indexes.
- BEA Table 2.4.5U (`U20405`) for monthly nominal expenditures.
- `API Keys.txt` for the BEA API key, while preferring local cached BEA files
  when they already exist.

Set `INFLATION_MEASURE` to `"yoy"` or `"mom"` in the settings cell below.


## 1. Settings


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import pickle
import time

import numpy as np
import pandas as pd

try:
    display
except NameError:
    display = print


def find_project_root() -> Path:
    # Find the project root whether the notebook is run from root or subdir.
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "market vs. nonmarket" / "supercore_lines.xlsx").exists():
            return candidate
        if candidate.name.lower() == "market vs. nonmarket" and (candidate / "supercore_lines.xlsx").exists():
            return candidate.parent
    return cwd


ROOT = find_project_root()
WORK_DIR = ROOT / "market vs. nonmarket"
CACHE = ROOT / "cache"
OUT = WORK_DIR / "output"
OUT.mkdir(parents=True, exist_ok=True)

SUPERCORE_LINES_PATH = WORK_DIR / "supercore_lines.xlsx"

# User input: choose "yoy" or "mom".
INFLATION_MEASURE = "yoy"

# For MoM only. False returns simple month-over-month percent changes; True
# annualizes each monthly rate.
ANNUALIZE_MOM = False

# Expenditure-weight timing:
# - "base_period": use expenditure shares from 12 months ago for YoY and from
#   the prior month for MoM.
# - "current": use current-month expenditure shares.
# - "average": use the average of base-period and current shares.
WEIGHT_TIMING = "base_period"

# If a line has a missing price change for a date, rescale the remaining
# available weights to sum to one. This keeps early or partial samples from
# understating the aggregate because of missing line-level data.
RENORMALIZE_WEIGHTS_FOR_AVAILABLE_LINES = True

START_YEAR = 1969
END_YEAR = datetime.today().year
REFRESH_BEA = False

BEA_DATASET = "NIUnderlyingDetail"
BEA_FREQUENCY = "M"
PRICE_TABLE = "U20404"  # Table 2.4.4U price indexes
SPEND_TABLE = "U20405"  # Table 2.4.5U expenditures

# "Less:" expenditure lines should enter the selected aggregate with a negative
# expenditure share. Line 338 is present in supercore_lines.xlsx.
NEGATIVE_SPEND_LINES = {338}

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)


## 2. BEA API and Cache Helpers


In [ ]:
def read_api_keys() -> dict[str, str]:
    # Read API keys without printing them.
    api = {}
    for path in [WORK_DIR / "API Keys.txt", ROOT / "API Keys.txt"]:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
            stripped = line.strip()
            if not stripped or stripped.startswith("#") or ":" not in stripped:
                continue
            key, value = stripped.split(":", 1)
            key = key.strip().upper()
            value = value.strip().strip('"').strip("'")
            if key and value and key not in api:
                api[key] = value
    return api


API = read_api_keys()


def as_float(value) -> float:
    text = str(value).replace(",", "").strip()
    if text in {"", ".", "-", "NA", "(NA)", "nan", "None"}:
        return np.nan
    return float(text)


def parse_bea_date(time_period, frequency=BEA_FREQUENCY) -> pd.Timestamp:
    tp = str(time_period)
    if frequency == "M":
        if "M" in tp:
            year, month = tp.split("M", 1)
            return pd.Timestamp(year=int(year), month=int(month), day=1)
        return pd.Timestamp(year=int(tp[:4]), month=int(tp[5:]), day=1)
    if frequency == "Q":
        if "Q" in tp:
            year, quarter = tp.split("Q", 1)
            return pd.Timestamp(year=int(year), month=3 * int(quarter) - 2, day=1)
        quarter = int(tp[-1])
        return pd.Timestamp(year=int(tp[:4]), month=3 * quarter - 2, day=1)
    raise ValueError(f"Unsupported BEA frequency: {frequency}")


def normalize_bea_rows(rows, frequency=BEA_FREQUENCY) -> pd.DataFrame:
    # Convert BEA API or cached rows to line/desc/date/value columns.
    if isinstance(rows, pd.DataFrame):
        df = rows.copy()
        if {"line", "desc", "date", "value"}.issubset(df.columns):
            df["date"] = pd.to_datetime(df["date"])
            return df[["line", "desc", "date", "value"]].copy()
        rows = df.to_dict("records")

    parsed = []
    for row in rows:
        line = row.get("LineNumber", row.get("line", row.get("Line")))
        desc = row.get("LineDescription", row.get("desc", row.get("Description", "")))
        value = row.get("DataValue", row.get("value"))
        time_period = row.get("TimePeriod", row.get("date"))
        if line is None or value is None or time_period is None:
            continue
        parsed.append(
            {
                "line": int(line),
                "desc": str(desc),
                "date": parse_bea_date(time_period, frequency),
                "value": as_float(value),
            }
        )
    return pd.DataFrame(parsed, columns=["line", "desc", "date", "value"])


def bea_cache_path(table: str, start_year=START_YEAR, end_year=END_YEAR) -> Path:
    return CACHE / f"bea_{BEA_DATASET}_{table}_{BEA_FREQUENCY}_{start_year}_{end_year}.csv"


def bea_cache_candidates(table: str) -> list[Path]:
    patterns = [
        f"bea_{BEA_DATASET}_{table}_{BEA_FREQUENCY}_*.csv",
        f"bea_{BEA_DATASET}_{table}_{BEA_FREQUENCY}_*.pkl",
        f"bea_NIPA_{table}_{BEA_FREQUENCY}_*.csv",
        f"bea_NIPA_{table}_{BEA_FREQUENCY}_*.pkl",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(CACHE.glob(pattern))
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)


def read_cached_bea_table(table: str, start_year=START_YEAR, end_year=END_YEAR) -> pd.DataFrame | None:
    # Reuse a local BEA cache when it covers the requested year range.
    start = pd.Timestamp(start_year, 1, 1)
    end = pd.Timestamp(end_year, 12, 31)

    exact = bea_cache_path(table, start_year, end_year)
    candidates = [exact] if exact.exists() else []
    candidates += [p for p in bea_cache_candidates(table) if p not in candidates]

    for path in candidates:
        try:
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path, parse_dates=["date"])
            else:
                df = normalize_bea_rows(pickle.loads(path.read_bytes()), BEA_FREQUENCY)
        except Exception:
            continue

        if not {"line", "desc", "date", "value"}.issubset(df.columns):
            continue

        df = df[["line", "desc", "date", "value"]].copy()
        df["date"] = pd.to_datetime(df["date"])
        if df["date"].min() > start:
            continue
        if df["date"].max().year < end_year and end_year < datetime.today().year:
            continue

        df = df[(df["date"] >= start) & (df["date"] <= end)].copy()
        if not df.empty:
            return df.sort_values(["line", "date"]).reset_index(drop=True)
    return None


def fetch_bea_table(table: str, start_year=START_YEAR, end_year=END_YEAR, retries=4) -> pd.DataFrame:
    # Fetch a BEA table, normalize it, and write a CSV cache.
    if "BEA" not in API:
        raise RuntimeError("BEA API key not found. Check market vs. nonmarket/API Keys.txt.")

    years = ",".join(str(y) for y in range(start_year, end_year + 1))
    params = {
        "UserID": API["BEA"],
        "method": "GetData",
        "DataSetName": BEA_DATASET,
        "TableName": table,
        "Frequency": BEA_FREQUENCY,
        "Year": years,
        "ResultFormat": "JSON",
    }
    url = "https://apps.bea.gov/api/data/?" + urlencode(params)

    last_error = None
    for attempt in range(retries):
        try:
            req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(req, timeout=180) as f:
                raw = json.loads(f.read().decode("utf-8"))
            results = raw.get("BEAAPI", {}).get("Results", {})
            if "Error" in results:
                err = results["Error"]
                raise RuntimeError(f"BEA API error {err.get('APIErrorCode')}: {err.get('APIErrorDescription')}")
            df = normalize_bea_rows(results.get("Data", []), BEA_FREQUENCY)
            df.to_csv(bea_cache_path(table, start_year, end_year), index=False)
            return df.sort_values(["line", "date"]).reset_index(drop=True)
        except Exception as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"Could not fetch BEA table {table}: {last_error}")


def bea_table(table: str, refresh=REFRESH_BEA) -> pd.DataFrame:
    cached = None if refresh else read_cached_bea_table(table)
    if cached is not None:
        return cached
    return fetch_bea_table(table)


## 3. Load Selected Lines and BEA Data


In [ ]:
def load_line_list(path: Path = SUPERCORE_LINES_PATH) -> pd.DataFrame:
    lines = pd.read_excel(path)
    lines.columns = [str(c).strip().lower() for c in lines.columns]
    if "line" not in lines.columns:
        raise ValueError("supercore_lines.xlsx must include a Line column.")
    item_col = "item" if "item" in lines.columns else lines.columns[-1]
    lines = lines.rename(columns={item_col: "item"})
    lines["line"] = lines["line"].astype(int)
    lines["item"] = lines["item"].astype(str).str.strip()
    return lines[["line", "item"]].drop_duplicates().sort_values("line").reset_index(drop=True)


def wide_by_line(df: pd.DataFrame, lines: list[int]) -> pd.DataFrame:
    out = (
        df[df["line"].isin(lines)]
        .pivot_table(index="date", columns="line", values="value", aggfunc="first")
        .sort_index()
    )
    return out.reindex(columns=lines)


line_list = load_line_list()
LINES = line_list["line"].tolist()

price_long = bea_table(PRICE_TABLE)
spend_long = bea_table(SPEND_TABLE)

prices = wide_by_line(price_long, LINES)
spending = wide_by_line(spend_long, LINES)

common_dates = prices.index.intersection(spending.index)
prices = prices.loc[common_dates]
spending = spending.loc[common_dates]

spending_adjusted = spending.copy()
for line in NEGATIVE_SPEND_LINES.intersection(spending_adjusted.columns):
    spending_adjusted[line] = -spending_adjusted[line]

coverage = line_list.set_index("line").copy()
coverage["price_nonmissing"] = prices.notna().sum()
coverage["spending_nonmissing"] = spending.notna().sum()
coverage["latest_weight"] = spending_adjusted.iloc[-1].div(spending_adjusted.iloc[-1].sum())

print(f"Loaded {len(LINES)} selected lines.")
print(f"BEA monthly sample: {common_dates.min().date()} to {common_dates.max().date()}")
display(coverage.head(15))


## 4. Construct Weighted Inflation


In [ ]:
def selected_inflation(
    price_indexes: pd.DataFrame | None = None,
    expenditure_values: pd.DataFrame | None = None,
    measure: str = INFLATION_MEASURE,
    weight_timing: str = WEIGHT_TIMING,
    annualize_mom: bool = ANNUALIZE_MOM,
    renormalize: bool = RENORMALIZE_WEIGHTS_FOR_AVAILABLE_LINES,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Return aggregate and line-level weighted inflation results.
    if price_indexes is None:
        price_indexes = prices
    if expenditure_values is None:
        expenditure_values = spending_adjusted

    measure = measure.lower().strip()
    if measure not in {"yoy", "mom"}:
        raise ValueError('measure must be "yoy" or "mom"')

    periods = 12 if measure == "yoy" else 1
    raw_change = price_indexes / price_indexes.shift(periods) - 1.0
    if measure == "mom" and annualize_mom:
        raw_change = (1.0 + raw_change) ** 12 - 1.0
    line_inflation_pct = 100.0 * raw_change

    shares = expenditure_values.div(expenditure_values.sum(axis=1, min_count=1), axis=0)
    if weight_timing == "base_period":
        weights = shares.shift(periods)
    elif weight_timing == "current":
        weights = shares
    elif weight_timing == "average":
        weights = 0.5 * (shares + shares.shift(periods))
    else:
        raise ValueError('weight_timing must be "base_period", "current", or "average"')

    valid = line_inflation_pct.notna() & weights.notna()
    effective_weights = weights.where(valid)
    if renormalize:
        effective_weights = effective_weights.div(effective_weights.sum(axis=1, min_count=1), axis=0)

    contributions_pp = effective_weights * line_inflation_pct
    aggregate = pd.DataFrame(index=price_indexes.index)
    aggregate["inflation_pct"] = contributions_pp.sum(axis=1, min_count=1)
    aggregate["weight_sum"] = effective_weights.sum(axis=1, min_count=1)
    aggregate["n_lines"] = valid.sum(axis=1)
    aggregate["selected_expenditure"] = expenditure_values.sum(axis=1, min_count=1)
    aggregate["measure"] = measure
    aggregate["weight_timing"] = weight_timing
    aggregate["annualized"] = bool(measure == "mom" and annualize_mom)
    aggregate = aggregate.dropna(subset=["inflation_pct"]).reset_index().rename(columns={"date": "date"})

    line_rows = []
    for line in price_indexes.columns:
        part = pd.DataFrame(
            {
                "date": price_indexes.index,
                "line": int(line),
                "price_index": price_indexes[line].to_numpy(),
                "expenditure": spending[line].to_numpy(),
                "adjusted_expenditure": expenditure_values[line].to_numpy(),
                "weight": effective_weights[line].to_numpy(),
                "line_inflation_pct": line_inflation_pct[line].to_numpy(),
                "contribution_pp": contributions_pp[line].to_numpy(),
            }
        )
        line_rows.append(part)
    line_detail = pd.concat(line_rows, ignore_index=True)
    line_detail = line_detail.merge(line_list, on="line", how="left")
    line_detail = line_detail[
        [
            "date",
            "line",
            "item",
            "price_index",
            "expenditure",
            "adjusted_expenditure",
            "weight",
            "line_inflation_pct",
            "contribution_pp",
        ]
    ].dropna(subset=["contribution_pp"])

    return aggregate, line_detail


aggregate, line_detail = selected_inflation()

suffix = INFLATION_MEASURE.lower()
aggregate_path = OUT / f"nonhousing_services_inflation_{suffix}.csv"
line_detail_path = OUT / f"nonhousing_services_line_contributions_{suffix}.csv"
aggregate.to_csv(aggregate_path, index=False)
line_detail.to_csv(line_detail_path, index=False)

print(f"Saved aggregate series to: {aggregate_path}")
print(f"Saved line contributions to: {line_detail_path}")
display(aggregate.tail(12))


## 5. Latest Contributors


In [ ]:
latest_date = aggregate["date"].max()
latest_lines = (
    line_detail[line_detail["date"].eq(latest_date)]
    .sort_values("contribution_pp", key=lambda s: s.abs(), ascending=False)
    .loc[:, ["date", "line", "item", "weight", "line_inflation_pct", "contribution_pp"]]
)

print(f"Latest observation: {latest_date:%Y-%m-%d}")
display(latest_lines.head(20))


## 6. Plot


In [ ]:
label = "YoY" if INFLATION_MEASURE.lower() == "yoy" else "MoM annualized" if ANNUALIZE_MOM else "MoM"

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(aggregate["date"], aggregate["inflation_pct"], color="#1f77b4", linewidth=1.8)
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
    ax.set_title(f"Weighted Non-Housing Services Inflation ({label})")
    ax.set_ylabel("Percent")
    ax.grid(True, axis="y", alpha=0.25)
    fig.autofmt_xdate()
    plt.show()
except ImportError:
    print("matplotlib is not installed in this Python environment; skipping the chart.")
